# 01 — Exploration du Steam Games Dataset (EDA manuelle)

**Projet :** `data-agent-pipeline`

**Objectif :** cette EDA constitue la **vérité terrain** du projet. Les valeurs calculées ici à la main serviront de tests de référence (`tests/test_ground_truth.py`) pour valider que la pipeline d'agents produit des réponses justes, et non des hallucinations.

**Question de référence :** *Quels sont les facteurs qui déterminent le succès commercial d'un jeu sur Steam, et quel positionnement recommanderiez-vous à un studio indépendant ?*

**Plan du notebook :**
1. Chargement et premier contact
2. Types et valeurs manquantes
3. Pièges spécifiques au dataset (owners, colonnes multi-valeurs, dates, free-to-play)
4. Distributions des proxys de succès
5. Vérité terrain : 6 questions calculées à la main
6. Création de l'échantillon 500 lignes
7. Bilan de l'exploration

---
## 0. Setup

Imports et options d'affichage. `max_columns = None` pour voir toutes les colonnes du dataset (il y en a beaucoup).

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


DATA_PATH = "../data/raw/games.csv"

---
## 1. Chargement et premier contact

Première prise de contact : dimensions, aperçu des premières lignes, liste des colonnes.

In [2]:
df = pd.read_csv(DATA_PATH)

print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
df.head(3)

Dimensions : 125,855 lignes × 39 colonnes


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],[],NaN,https://shared.akamai.steamstatic.com/store_it...,NaN,NaN,NaN,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],[],NaN,https://shared.akamai.steamstatic.com/store_it...,http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,True,False,False,0,NaN,0,252,3,NaN,0,231,NaN,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",[],NaN,https://shared.akamai.steamstatic.com/store_it...,https://www.facebook.com/8FloorGames/,https://www.facebook.com/8FloorGames,support@8floor.net,True,True,False,0,NaN,0,21,3,NaN,0,0,NaN,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN


In [3]:
# Liste complète des colonnes (à comparer avec ce que détectera le futur Profiler)
list(df.columns)

['AppID',
 'Name',
 'Release date',
 'Estimated owners',
 'Peak CCU',
 'Required age',
 'Price',
 'DiscountDLC count',
 'About the game',
 'Supported languages',
 'Full audio languages',
 'Reviews',
 'Header image',
 'Website',
 'Support url',
 'Support email',
 'Windows',
 'Mac',
 'Linux',
 'Metacritic score',
 'Metacritic url',
 'User score',
 'Positive',
 'Negative',
 'Score rank',
 'Achievements',
 'Recommendations',
 'Notes',
 'Average playtime forever',
 'Average playtime two weeks',
 'Median playtime forever',
 'Median playtime two weeks',
 'Developers',
 'Publishers',
 'Categories',
 'Genres',
 'Tags',
 'Screenshots',
 'Movies']

In [4]:
print(len(df.columns))          # 39 ou 40 ?
df[["Price", "DiscountLC count" if False else df.columns[7], "About the game"]].head(3)

39


,Price,DiscountDLC count,About the game
2539430,0,0,NaN
496350,65,0,"Springtime, April: when the cherry trees come ..."
1034400,0,0,"Immerse yourself in the most beloved, mystical..."


In [5]:
COLS = ["AppID", "Name", "Release date", "Estimated owners", "Peak CCU",
        "Required age", "Price", "Discount", "DLC count", "About the game",
        "Supported languages", "Full audio languages", "Reviews", "Header image",
        "Website", "Support url", "Support email", "Windows", "Mac", "Linux",
        "Metacritic score", "Metacritic url", "User score", "Positive", "Negative",
        "Score rank", "Achievements", "Recommendations", "Notes",
        "Average playtime forever", "Average playtime two weeks",
        "Median playtime forever", "Median playtime two weeks",
        "Developers", "Publishers", "Categories", "Genres", "Tags",
        "Screenshots", "Movies"]

df = pd.read_csv(DATA_PATH, skiprows=1, names=COLS)
print(df.shape)  # attendu : (~125000, 40)

(125855, 40)


In [9]:
df[["AppID", "Name", "Release date", "Price", "Discount", "DLC count"]].head(5)

,AppID,Name,Release date,Price,Discount,DLC count
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0.00,0,0
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",5.24,65,0
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",4.99,0,0
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",8.99,0,1
4,3631080,Maze Quest VR,"Apr 24, 2025",4.99,0,0


In [10]:
df["Price"].describe()

count   125,855.00
mean          4.81
std          12.49
min           0.00
25%           0.59
50%           2.39
75%           5.54
max         999.98
Name: Price, dtype: float64

### 📝 À noter — Section 1

> - Nombre de jeux : `125,855`
> - Nombre de colonnes : `39`
> - Grain du dataset : une ligne = un jeu (vérifier l'unicité de `AppID` en section 2)
> - Colonnes qui semblent inutiles pour la problématique : `...`

---
## 2. Types et valeurs manquantes

Objectif : classifier les colonnes (identifiant / dimension / métrique / temporelle / texte) et repérer les colonnes inutilisables car trop trouées.

**Seuils d'alerte :** > 5 % de manquants = à surveiller · > 20 % = alerte · > 50 % = probablement inutilisable sans traitement.

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 125855 entries, 0 to 125854
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       125855 non-null  int64  
 1   Name                        125854 non-null  str    
 2   Release date                125855 non-null  str    
 3   Estimated owners            125855 non-null  str    
 4   Peak CCU                    125855 non-null  int64  
 5   Required age                125855 non-null  int64  
 6   Price                       125855 non-null  float64
 7   Discount                    125855 non-null  int64  
 8   DLC count                   125855 non-null  int64  
 9   About the game              117393 non-null  str    
 10  Supported languages         125855 non-null  str    
 11  Full audio languages        125855 non-null  str    
 12  Reviews                     12181 non-null   str    
 13  Header image             

In [12]:
# Taux de valeurs manquantes par colonne, trié
missing = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
missing[missing > 0]

Movies           100.00
Score rank        99.97
Metacritic url    96.62
Reviews           90.32
Notes             81.48
Website           59.86
Support url       56.06
Tags              33.77
Support email     17.98
Categories         7.12
Publishers         7.09
About the game     6.72
Developers         6.71
Genres             6.69
Screenshots        4.79
Header image       0.06
dtype: float64

In [13]:
# Vérification de l'unicité de la clé primaire
print(f"AppID uniques : {df['AppID'].nunique():,} / {len(df):,} lignes")
print(f"Doublons éventuels : {df['AppID'].duplicated().sum()}")

AppID uniques : 125,855 / 125,855 lignes
Doublons éventuels : 0


### 📝 À noter — Section 2

> - Top 5 des colonnes les plus manquantes : `Movies, Score rank, Metacritic url, Reviews, Notes`
> - Colonnes à écarter (> 50 % manquants) : `...`
> - `AppID` est-il bien unique ? `oui`
> - Colonnes stockées en texte alors qu'elles contiennent des données structurées : `Estimated owners`, `Release date`, `Genres`, `Tags`, `Categories`, ... (traitées en section 3)
>
> ⚠️ Le futur **Profiler** devra détecter ces mêmes points automatiquement — cette liste est un point de comparaison.

---
## 3. Les pièges spécifiques au dataset Steam

Trois colonnes clés ne sont **pas directement exploitables** telles quelles. Ce sont exactement les subtilités que le futur Analyste devra gérer — chaque piège documenté ici est un cas de test potentiel.

### 3a. `Estimated owners` — une plage en texte, pas un nombre

La colonne contient des chaînes du type `"20000 - 50000"`. Impossible de faire une moyenne dessus sans conversion. On crée `owners_mid`, le point médian de la plage.

In [17]:
df["Estimated owners"].value_counts().head(100)

Estimated owners
0 - 20000                75705
0 - 0                    24579
20000 - 50000            11397
50000 - 100000            5355
100000 - 200000           3454
200000 - 500000           2856
500000 - 1000000          1155
1000000 - 2000000          729
2000000 - 5000000          405
5000000 - 10000000         125
10000000 - 20000000         51
20000000 - 50000000         31
50000000 - 100000000         9
100000000 - 200000000        4
Name: count, dtype: int64

In [52]:
df.loc[df["Estimated owners"] == "0 - 0", "release_dt"].describe()

count                         24579
mean     2024-08-21 18:39:03.756865
min             2008-10-23 00:00:00
25%             2023-10-16 00:00:00
50%             2025-03-12 00:00:00
75%             2025-10-17 00:00:00
max             2026-12-01 00:00:00
Name: release_dt, dtype: object

In [53]:
mask00 = df["Estimated owners"] == "0 - 0"
part_futur = (df.loc[mask00, "release_dt"] > pd.Timestamp("2026-07-15")).mean() * 100
print(f"Part des jeux 0-0 non encore sortis : {part_futur:.1f} %")

Part des jeux 0-0 non encore sortis : 0.0 %


In [15]:
def owners_midpoint(s):
    """Convertit une plage '20000 - 50000' en point médian (35000.0)."""
    if pd.isna(s):
        return np.nan
    low, high = s.replace(",", "").split(" - ")
    return (int(low) + int(high)) / 2

df["owners_mid"] = df["Estimated owners"].apply(owners_midpoint)
df["owners_mid"].describe()

count       125,855.00
mean         83,548.69
std       1,273,025.26
min               0.00
25%          10,000.00
50%          10,000.00
75%          10,000.00
max     150,000,000.00
Name: owners_mid, dtype: float64

In [32]:
df["review_total"] = df["Positive"] + df["Negative"]
df["review_ratio"] = df["Positive"] / df["review_total"].replace(0, np.nan)

In [16]:
# Cas particulier : la tranche '0 - 20000' (point médian 10000) domine-t-elle ?
part_basse = (df["Estimated owners"] == "0 - 20000").mean() * 100
print(f"Part des jeux dans la tranche la plus basse (0 - 20000) : {part_basse:.1f} %")

Part des jeux dans la tranche la plus basse (0 - 20000) : 60.2 %


### 📝 À noter — Section 3a

> - Nombre de tranches distinctes : `14`
> - Part des jeux dans la tranche la plus basse : `60.2 %`
> - Limite méthodologique à mentionner dans le rapport final : `owners_mid` est un **proxy grossier** (point médian d'une estimation par tranche), pas une mesure exacte des ventes.
> - Hypothèse émise (pré-releases) → vérifiée sur les données → partiellement réfutée → interprétation corrigée.

### 3b. `Genres`, `Categories`, `Tags` — colonnes multi-valeurs

Ces colonnes contiennent plusieurs valeurs séparées par des virgules (`"Action,Indie,RPG"`). Toute statistique « par genre » exige un `explode()`. On construit un dataframe explosé réutilisable : `df_genres`.

⚠️ Après explosion, un jeu apparaît sur plusieurs lignes : ne **jamais** sommer des métriques globales sur `df_genres` sans en avoir conscience (double comptage).

In [35]:
df_genres = (
    df.assign(genre=df["Genres"].str.split(","))
      .explode("genre")
)
df_genres["genre"] = df_genres["genre"].str.strip()

print(f"Lignes avant explosion : {len(df):,} — après : {len(df_genres):,}")
df_genres["genre"].value_counts().head(15)

Lignes avant explosion : 125,855 — après : 347,006


genre
Indie                    82884
Casual                   51677
Action                   47344
Adventure                46293
Simulation               24912
Strategy                 23058
RPG                      21577
Free To Play             12605
Early Access             11572
Sports                    4971
Racing                    4169
Massively Multiplayer     2596
Utilities                 1091
Design & Illustration      584
Animation & Modeling       484
Name: count, dtype: int64

In [36]:
# Nombre de genres distincts et jeux sans genre
print(f"Genres distincts : {df_genres['genre'].nunique()}")
print(f"Jeux sans genre renseigné : {df['Genres'].isna().sum():,}")

Genres distincts : 33
Jeux sans genre renseigné : 8,423


### 📝 À noter — Section 3b

> - Top 5 des genres par nombre de jeux : `Indie, Casual, ACtion, Adventure, Simulation`
> - Nombre de genres distincts : `33`
> - Jeux sans genre : `8423`
> - Même logique à appliquer plus tard sur `Tags` (granularité plus fine, utile pour le positionnement indé).

### 3c. `Release date` — du texte à parser

Formats hétérogènes (parfois seulement une année). On parse avec `errors="coerce"` et on quantifie la perte.

In [37]:
df["release_dt"] = pd.to_datetime(df["Release date"], errors="coerce", format="mixed")
df["release_year"] = df["release_dt"].dt.year

nb_na = df["release_dt"].isna().sum()
print(f"Dates non parsées : {nb_na:,} ({nb_na / len(df) * 100:.2f} %)")
print(f"Plage temporelle : {df['release_dt'].min()} → {df['release_dt'].max()}")

Dates non parsées : 0 (0.00 %)
Plage temporelle : 1997-06-30 00:00:00 → 2026-12-01 00:00:00


In [38]:
# Sorties par année — sanity check visuel rapide
df["release_year"].value_counts().sort_index().tail(20)

release_year
2007       90
2008      158
2009      321
2010      255
2011      260
2012      322
2013      468
2014     1523
2015     2515
2016     4141
2017     5922
2018     7461
2019     7243
2020     8804
2021    11067
2022    12288
2023    14598
2024    20033
2025    24973
2026     3316
Name: count, dtype: int64

### 📝 À noter — Section 3c

> - % de dates non parsées : `0 %`
> - Plage temporelle couverte : `1997-06-30 00:00:00 → 2026-12-01 00:00:00`
> - Anomalies (dates futures ? années aberrantes ?) : `0`
> - ⚠️ La dernière année est probablement **incomplète** (dataset figé en cours d'année) : ne pas conclure à une « chute des sorties ».

### 3d. `Price` — le piège des free-to-play

Les jeux à 0 € (free-to-play) faussent tout prix moyen. Il faut les quantifier et les traiter explicitement — c'est le genre de subtilité qu'un rapport « client » doit mentionner.

In [39]:
part_f2p = (df["Price"] == 0).mean() * 100
print(f"Jeux gratuits : {part_f2p:.1f} % du dataset")

print("\nDistribution du prix — tous jeux :")
print(df["Price"].describe())

print("\nDistribution du prix — jeux payants uniquement :")
print(df.loc[df["Price"] > 0, "Price"].describe())

print("\nPrix médian — jeux payants uniquement :")
print(df.loc[df["Price"] > 0, "Price"].median())

Jeux gratuits : 21.2 % du dataset

Distribution du prix — tous jeux :
count   125,855.00
mean          4.81
std          12.49
min           0.00
25%           0.59
50%           2.39
75%           5.54
max         999.98
Name: Price, dtype: float64

Distribution du prix — jeux payants uniquement :
count   99,194.00
mean         6.10
std         13.78
min          0.49
25%          1.49
50%          3.59
75%          6.99
max        999.98
Name: Price, dtype: float64

Prix médian — jeux payants uniquement :
3.59


### 📝 À noter — Section 3d

> - Part des free-to-play : `21.2 %` *(→ vérité terrain n° 4)*
> - Prix médian des jeux payants : `3.59$`
> - Prix maximum (aberrant ?) : `999.98$, pas aberrant`
> - Décision méthodologique : pour les analyses de prix, travailler sur les **jeux payants** et le préciser dans chaque résultat.

---
## 4. Distributions des proxys de succès

La problématique porte sur le **succès commercial**. Le dataset n'a pas de chiffre de ventes : on construit des proxys.

- `owners_mid` → volume (déjà créé en 3a)
- `review_total = Positive + Negative` → visibilité / engagement
- `review_ratio = Positive / review_total` → qualité perçue

In [40]:
df[["Price", "owners_mid", "review_total", "review_ratio"]].describe(
    percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
)

,Price,owners_mid,review_total,review_ratio
count,"125,855.00","125,855.00","125,855.00","82,956.00"
mean,4.81,"83,548.69","1,183.06",0.76
std,12.49,"1,273,025.26","32,214.01",0.24
min,0.00,0.00,0.00,0.00
25%,0.59,"10,000.00",0.00,0.65
50%,2.39,"10,000.00",6.00,0.82
75%,5.54,"10,000.00",46.00,0.94
95%,14.99,"150,000.00","1,284.00",1.00
99%,29.99,"1,500,000.00","15,165.28",1.00
max,999.98,"150,000,000.00","8,815,087.00",1.00


In [41]:
# Asymétrie de la distribution : moyenne vs médiane
for col in ["owners_mid", "review_total"]:
    print(f"{col:>15} — moyenne : {df[col].mean():>14,.0f} | médiane : {df[col].median():>10,.0f}")

     owners_mid — moyenne :         83,549 | médiane :     10,000
   review_total — moyenne :          1,183 | médiane :          6


In [42]:
# Concentration : quelle part des propriétaires est captée par le top 1 % des jeux ?
seuil_top1 = df["owners_mid"].quantile(0.99)
part_top1 = df.loc[df["owners_mid"] >= seuil_top1, "owners_mid"].sum() / df["owners_mid"].sum() * 100
print(f"Le top 1 % des jeux concentre {part_top1:.1f} % des propriétaires estimés")

Le top 1 % des jeux concentre 62.5 % des propriétaires estimés


### 📝 À noter — Section 4

> - Écart moyenne / médiane sur `owners_mid` : `73549` → distribution en **loi de puissance** (une poignée de hits, une immense majorité de jeux invisibles)
> - Part des propriétaires captée par le top 1 % : `62.5 %`
> - **Conséquence méthodologique :** préférer la **médiane** à la moyenne partout dans le rapport
> - **Conséquence business :** constat central pour la recommandation au studio indé (le marché est hyper-concentré)

---
## 5. Vérité terrain — 6 questions calculées à la main

⭐ **Section la plus importante du notebook.** Chaque valeur calculée ici deviendra une assertion dans `tests/test_ground_truth.py`. Reporter les résultats **exacts** dans le tableau récapitulatif en fin de section.

### Q1 — Prix médian par genre (top 10 genres, jeux payants)

In [43]:
top10_genres = df_genres["genre"].value_counts().head(10).index

q1 = (
    df_genres[(df_genres["genre"].isin(top10_genres)) & (df_genres["Price"] > 0)]
    .groupby("genre")["Price"]
    .median()
    .sort_values(ascending=False)
)
q1

genre
Early Access   6.39
RPG            4.99
Strategy       3.99
Adventure      3.99
Simulation     3.99
Sports         3.99
Action         3.74
Free To Play   3.49
Indie          2.99
Casual         2.99
Name: Price, dtype: float64

### Q2 — Nombre de sorties par année depuis 2010

In [44]:
q2 = (
    df.loc[df["release_year"] >= 2010, "release_year"]
    .value_counts()
    .sort_index()
    .astype(int)
)
q2

release_year
2010      255
2011      260
2012      322
2013      468
2014     1523
2015     2515
2016     4141
2017     5922
2018     7461
2019     7243
2020     8804
2021    11067
2022    12288
2023    14598
2024    20033
2025    24973
2026     3316
Name: count, dtype: int64

### Q3 — Genre au meilleur `review_ratio` médian (genres ≥ 500 jeux)

In [45]:
genres_500 = df_genres["genre"].value_counts()
genres_500 = genres_500[genres_500 >= 500].index

q3 = (
    df_genres[df_genres["genre"].isin(genres_500)]
    .groupby("genre")["review_ratio"]
    .median()
    .sort_values(ascending=False)
)
q3.head(10)

genre
Casual         0.83
Indie          0.83
Adventure      0.81
Action         0.81
RPG            0.80
Strategy       0.80
Free To Play   0.79
Racing         0.79
Early Access   0.78
Sports         0.77
Name: review_ratio, dtype: float64

### Q4 — Part des jeux gratuits (%)

In [46]:
q4 = (df["Price"] == 0).mean() * 100
print(f"Part des jeux gratuits : {q4:.2f} %")

Part des jeux gratuits : 21.18 %


### Q5 — % de jeux dépassant 50 000 owners, par tranche de prix

In [51]:
payants = df[df["Price"] > 0].copy()
payants["tranche_prix"] = np.where(payants["Price"] < 15, "< 15 $", ">= 15 $")

q5 = payants.groupby("tranche_prix")["owners_mid"].apply(lambda s: (s > 50_000).mean() * 100)
q5  # % de jeux dépassant 50 000 owners, par tranche de prix

tranche_prix
< 15 $    11.61
>= 15 $   17.02
Name: owners_mid, dtype: float64

### Q6 — Top 5 des genres par total de propriétaires estimés

⚠️ Ici la somme sur `df_genres` est assumée : un jeu multi-genres compte dans chacun de ses genres (à documenter comme choix méthodologique).

In [48]:
q6 = (
    df_genres.groupby("genre")["owners_mid"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
)
q6

genre
Action         6,677,200,000.00
Indie          4,683,355,000.00
Adventure      4,451,130,000.00
RPG            2,785,510,000.00
Free To Play   2,287,930,000.00
Name: owners_mid, dtype: float64

### 📝 Tableau récapitulatif — Vérité terrain

À remplir avec les valeurs exactes des sorties ci-dessus. Ce tableau alimente directement `tests/test_ground_truth.py`.

| # | Question | Réponse calculée | Tolérance test |
|---|----------|------------------|----------------|
| Q1 | Prix médian du genre `Early Access` (1er du top 10) | `6.39 $` | ± 0.01 |
| Q2 | Sorties en 2023 (exemple d'année) | `14598` | exact |
| Q3 | Genre au meilleur review_ratio médian (≥ 500 jeux) | `Casual` (`0.83`) | exact / ± 0.01 |
| Q4 | Part des jeux gratuits | `21.18 %` | ± 0.1 |
| Q5 | Part des jeux > 50 000 owners : < 15 $ vs ≥ 15 $ (payants) | `11.61 %` vs `17.02 %` | ± 0.1 |
| Q6 | 1er genre par propriétaires cumulés | `Action` (`6,677,200,000.00`) | exact |

**Choix méthodologiques associés (à réinjecter dans les prompts des agents) :**
- Prix : jeux payants uniquement, sauf mention contraire
- Owners : point médian de la plage (`owners_mid`)
- Stats par genre : dataframe explosé, un jeu multi-genres compte dans chaque genre
- Agrégats : médiane privilégiée (distributions en loi de puissance)

---
## 6. Création de l'échantillon 500 lignes (étape 11)

Échantillon reproductible (`random_state=42`) committé dans `data/samples/` — c'est lui que liront les scripts de validation du SDK (phase 2). Le dataset complet reste dans `data/raw/` (gitignored).

In [49]:
sample = df.sample(500, random_state=42)
sample.to_csv("../data/samples/steam_sample_500.csv", index=False)
print(f"Échantillon sauvegardé : {sample.shape[0]} lignes × {sample.shape[1]} colonnes")

Échantillon sauvegardé : 500 lignes × 45 colonnes


---
## 7. Bilan de l'exploration



> `### Synthèse de l'exploration

Le dataset couvre **125 855 jeux** et **40 colonnes**, avec un piège structurel détecté dès le
chargement : un header défectueux (39 noms pour 40 valeurs, colonnes `Discount` et `DLC count`
fusionnées) qui décalait silencieusement toutes les colonnes — corrigé par un chargement avec
noms explicites (`skiprows=1, names=COLS`). Sans cette vérification, toutes les statistiques
de prix auraient en réalité porté sur des pourcentages de remise.

Trois familles de colonnes exigent un pré-traitement systématique : `Estimated owners` est une
plage textuelle convertie en point médian (`owners_mid`, proxy grossier à assumer) ;
`Genres`/`Categories`/`Tags` sont multi-valeurs et nécessitent une explosion (un jeu
multi-genres compte dans chacun de ses genres — choix documenté) ; les dates se parsent sans perte (0 % d'échec) mais s'étendent jusqu'à fin 2026 : le dataset
inclut des jeux non encore sortis. Ils n'expliquent toutefois qu'une minorité des 19,5 % de
jeux à zéro propriétaire (`0 - 0`) : la médiane de sortie de cette tranche est mars 2025 —
il s'agit majoritairement de sorties récentes à traction quasi nulle, pour lesquelles
l'estimation d'owners est indisponible. La tranche `0 - 0` est donc à traiter comme
« donnée manquante ou négligeable », pas comme une mesure..

Le constat central : le marché Steam suit une **loi de puissance**. 79,7 % des jeux plafonnent
sous les 20 000 propriétaires (dont 19,5 % à zéro), tandis que le top 1 % concentre à lui seul
**62,5 %** des propriétaires estimés. Conséquences méthodologiques : la **médiane** remplace la
moyenne partout, les analyses de prix excluent les free-to-play (21,18 % du catalogue — un
majorant, `Price == 0` incluant des prix non renseignés), et la granularité par tranches
d'`owners_mid` rendant la médiane non discriminante entre segments de prix, la question Q5 a
été reformulée en taux de dépassement de seuil : **11,61 %** des jeux < 15 $ dépassent 50 000
owners, contre **17,02 %** des jeux ≥ 15 $ (corrélation, non causalité).

Six valeurs de vérité terrain ont été figées (tableau section 5) : genre au prix médian le plus
élevé du top 10 (Early Access, 6,39 $ — un statut de commercialisation que Steam range dans
`Genres`), 14 598 sorties en 2023, meilleur ratio d'avis positifs médian (Casual, 0,83), part
de F2P (21,18 %), taux de percée par segment de prix (11,61 % vs 17,02 %), et genre dominant en
owners cumulés (Action, ≈ 6,68 Md, tolérance ± 1 %). Elles constituent la base de
`tests/test_ground_truth.py` : toute réponse de la pipeline divergeant de ces valeurs signalera
une régression ou une hallucination. Vérité terrain calculée sur le dataset complet, pré-releases
incluses.`

### Checklist de sortie de phase 1

- [ ] Dimensions et grain du dataset notés (section 1)
- [ ] Colonnes inutilisables identifiées (section 2)
- [ ] Les 4 pièges documentés : owners, multi-valeurs, dates, F2P (section 3)
- [ ] Asymétrie des distributions constatée → médiane partout (section 4)
- [ ] Tableau de vérité terrain rempli avec 6 valeurs exactes (section 5)
- [ ] Échantillon 500 lignes créé et committé (section 6)